In [2]:
import time
import difflib
import pandas as pd
from tqdm import tqdm

In [3]:
sales = pd.read_csv("data/raw/video_games_sales.csv")
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16717 non-null  object 
 1   Platform         16719 non-null  object 
 2   Year_of_Release  16450 non-null  float64
 3   Genre            16717 non-null  object 
 4   Publisher        16665 non-null  object 
 5   NA_Sales         16719 non-null  float64
 6   EU_Sales         16719 non-null  float64
 7   JP_Sales         16719 non-null  float64
 8   Other_Sales      16719 non-null  float64
 9   Global_Sales     16719 non-null  float64
 10  Critic_Score     8137 non-null   float64
 11  Critic_Count     8137 non-null   float64
 12  User_Score       10015 non-null  object 
 13  User_Count       7590 non-null   float64
 14  Developer        10096 non-null  object 
 15  Rating           9950 non-null   object 
dtypes: float64(9), object(7)
memory usage: 2.0+ MB


## Fixing dev name

In [47]:
indie_devs = pd.read_csv("data/raw/indie_games_developers.csv")
other_devs = pd.read_csv("data/raw/video_games_developers.csv")
devs = pd.concat([indie_devs, other_devs[indie_devs.columns]]).drop_duplicates(subset='Developer')
devs.to_csv("data/processed/unified_videogames_devs_and_publishers.csv")
devs.info()

<class 'pandas.core.frame.DataFrame'>
Index: 830 entries, 0 to 685
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Developer        830 non-null    object
 1   City             813 non-null    object
 2   Autonomous area  412 non-null    object
 3   Country          830 non-null    object
 4   Notable games    751 non-null    object
 5   Notes            444 non-null    object
dtypes: object(6)
memory usage: 45.4+ KB


In [4]:
unmatched_developers = sales[~sales["Developer"].isin(devs["Developer"])]["Developer"].unique()
print(len(unmatched_developers))

1383


In [ ]:
no_match = []
skipped = []
dev_list = devs["Developer"].values
for unmatched_dev in tqdm(unmatched_developers):
    if pd.notna(unmatched_dev) and unmatched_dev not in skipped:
        time.sleep(0.25)
        match = difflib.get_close_matches(unmatched_dev, dev_list, n=1, cutoff=0.6)
        try:
            if match[0] and input(unmatched_dev + " -> " + match[0]) == "y":
                sales.loc[sales["Developer"] == unmatched_dev, "Developer"] = match[0]
            else:
                skipped.append(unmatched_dev)
        except IndexError as e:
            no_match.append(unmatched_dev)

In [ ]:
for skipped_dev in tqdm(skipped):
    time.sleep(0.25)
    match = difflib.get_close_matches(skipped_dev, dev_list, n=1, cutoff=0.75)
    try:
        if match[0] and input(skipped_dev + " -> " + match[0]) == "y":
            sales.loc[sales["Developer"] == skipped_dev, "Developer"] = match[0]
    except IndexError as e:
            no_match.append(unmatched_dev)

In [ ]:
sales.to_csv("./data/processed/processed_video_game_sales.csv")

## Adding the location of developer/publisher to sales data

### Filling null devs with publisher

In [26]:
processed_devs = pd.read_csv("data/processed/unified_videogames_devs_and_publishers.csv")
processed_sales = pd.read_csv("data/processed/processed_video_game_sales.csv")
processed_sales['Developer'] = processed_sales['Developer'].fillna(processed_sales['Publisher'])
processed_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       16719 non-null  int64  
 1   Name             16717 non-null  object 
 2   Platform         16719 non-null  object 
 3   Year_of_Release  16450 non-null  float64
 4   Genre            16717 non-null  object 
 5   Publisher        16665 non-null  object 
 6   NA_Sales         16719 non-null  float64
 7   EU_Sales         16719 non-null  float64
 8   JP_Sales         16719 non-null  float64
 9   Other_Sales      16719 non-null  float64
 10  Global_Sales     16719 non-null  float64
 11  Critic_Score     8137 non-null   float64
 12  Critic_Count     8137 non-null   float64
 13  User_Score       10015 non-null  object 
 14  User_Count       7590 non-null   float64
 15  Developer        16674 non-null  object 
 16  Rating           9950 non-null   object 
dtypes: float64(9

In [27]:
result = processed_sales.merge(
    processed_devs[['Developer', 'Country']],
    on='Developer',
    how='left',
    validate='m:1'
)

result = result.rename(columns={'Country': 'Developer/Publisher country'})
result.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0                   16719 non-null  int64  
 1   Name                         16717 non-null  object 
 2   Platform                     16719 non-null  object 
 3   Year_of_Release              16450 non-null  float64
 4   Genre                        16717 non-null  object 
 5   Publisher                    16665 non-null  object 
 6   NA_Sales                     16719 non-null  float64
 7   EU_Sales                     16719 non-null  float64
 8   JP_Sales                     16719 non-null  float64
 9   Other_Sales                  16719 non-null  float64
 10  Global_Sales                 16719 non-null  float64
 11  Critic_Score                 8137 non-null   float64
 12  Critic_Count                 8137 non-null   float64
 13  User_Score      

### Swapping devs with null location for the publisher

In [28]:
where_location_is_null = result['Developer/Publisher country'].isna()
result.loc[where_location_is_null, "Developer"] = result.loc[where_location_is_null, 'Publisher']
result = result.drop(columns=["Developer/Publisher country"])
result = result.merge(
    processed_devs[['Developer', 'Country']],
    on='Developer',
    how='left'
)
result = result.rename(columns={'Country': 'Developer/Publisher country'})
result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0                   16719 non-null  int64  
 1   Name                         16717 non-null  object 
 2   Platform                     16719 non-null  object 
 3   Year_of_Release              16450 non-null  float64
 4   Genre                        16717 non-null  object 
 5   Publisher                    16665 non-null  object 
 6   NA_Sales                     16719 non-null  float64
 7   EU_Sales                     16719 non-null  float64
 8   JP_Sales                     16719 non-null  float64
 9   Other_Sales                  16719 non-null  float64
 10  Global_Sales                 16719 non-null  float64
 11  Critic_Score                 8137 non-null   float64
 12  Critic_Count                 8137 non-null   float64
 13  User_Score      

### Dropping rows with null location

In [29]:
final_dataset = result.dropna(subset='Developer/Publisher country')
final_dataset.to_csv("data/final/final_videogame_sales.csv")
final_dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11148 entries, 0 to 16716
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0                   11148 non-null  int64  
 1   Name                         11146 non-null  object 
 2   Platform                     11148 non-null  object 
 3   Year_of_Release              10992 non-null  float64
 4   Genre                        11146 non-null  object 
 5   Publisher                    11144 non-null  object 
 6   NA_Sales                     11148 non-null  float64
 7   EU_Sales                     11148 non-null  float64
 8   JP_Sales                     11148 non-null  float64
 9   Other_Sales                  11148 non-null  float64
 10  Global_Sales                 11148 non-null  float64
 11  Critic_Score                 6782 non-null   float64
 12  Critic_Count                 6782 non-null   float64
 13  User_Score           

## Treating Genres

In [30]:
final_dataset = pd.read_csv("data/final/final_videogame_sales.csv")
final_dataset["Genre"].unique()

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', nan], dtype=object)

In [31]:
final_dataset.loc[final_dataset['Genre'].isna(), "Genre"] = "Other"
final_dataset.loc[final_dataset['Genre'] == "Misc", "Genre"] = "Other"
final_dataset["Genre"].unique()

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Other',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy'], dtype=object)

In [32]:
final_dataset.to_csv("data/final/final_videogame_sales.csv")